In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")

print("Path to dataset files:", path)

Using Colab cache for faster access to the '140k-real-and-fake-faces' dataset.
Path to dataset files: /kaggle/input/140k-real-and-fake-faces


In [ ]:
from google.colab import drive
import os

print("Đang kết nối Google Drive...")
drive.mount('/content/drive')

Đang kết nối Google Drive...
Mounted at /content/drive


In [ ]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 24.3 MB/s eta 0:00:00


In [ ]:

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
# import lpips
import numpy as np
import matplotlib.pyplot as plt

class Config:
    # Đường dẫn dữ liệu
    TRAIN_REAL_DIR = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real"
    # TRAIN_FAKE_DIR = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/fake"
    VAL_REAL_DIR = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid/real"
    # VAL_FAKE_DIR = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid/fake"

    # Đường dẫn lưu model
    DRIVE_PATH = "/content/drive/MyDrive/final/stegano_fix_4.4"
    SAVE_DIR = DRIVE_PATH

    # Tham số huấn luyện

    IMG_SIZE = 128
    BATCH_SIZE = 32
    NUM_EPOCHS = 40
    LEARNING_RATE = 1e-3
    # LEARNING_RATE_DISC = 1e-4
    BASE_CHANNEL = 16

    # Trọng số Loss (Composite Loss)
    LAMBDA_ADV = 0.00
    LAMBDA_SECRET_L1 = 1.0
    LAMBDA_SECRET_L2 = 0.5
    LAMBDA_COVER_L1 = 1.0
    LAMBDA_COVER_L2 = 0.5
    LAMBDA_COVER_LPIPS = 0.0

    # Thiết bị
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tạo thư mục lưu model
os.makedirs(Config.SAVE_DIR, exist_ok=True)
print(f"Cấu hình đã sẵn sàng. Thiết bị: {Config.DEVICE}")

Cấu hình đã sẵn sàng. Thiết bị: cuda


In [ ]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        # Depthwise: Gom nhóm theo channels
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride, padding, groups=in_channels, bias=False)
        # Pointwise: Trộn thông tin giữa các channels
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

class Encoder(nn.Module):
    def __init__(self, base_c = 16):
        super().__init__()

        def conv_block(in_c, out_c, stride=1):
            return nn.Sequential(
                SeparableConv2d(in_c, out_c, stride=stride),
                nn.BatchNorm2d(out_c),
                nn.LeakyReLU(0.1, inplace=True))

        def conv_block_no_relu(in_c, out_c, stride=1):
            return nn.Sequential(
                SeparableConv2d(in_c, out_c, stride=stride))


        self.head = conv_block(6, base_c) # 128x128x16
        self.down1 = conv_block(base_c, base_c*2, stride=2) # 64x64x32
        self.down2 = conv_block(base_c*2, base_c*4, stride=2) #  32x32x64
        self.down3 = conv_block(base_c*4, base_c*8, stride=2) #  16x16x128

        self.bottleneck = conv_block(base_c*8, base_c*8, stride=2) # 8x8x128
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

        self.up1 = conv_block(base_c*8 + base_c*8, base_c*4)  # 16X16x
        self.up2 = conv_block(base_c*4 + base_c*4, base_c*2) # 32x32x
        self.up3 = conv_block(base_c*2 + base_c*2, base_c) # 64x64x
        self.up4 = conv_block(base_c + base_c, base_c) # 128x128xbase_c
        # self.tail = nn.Tanh()
        # self.up4 = nn.Conv2d(base_c + base_c, 3, )
        #self.tail = nn.Sequential(nn.Conv2d(base_c, 3, kernel_size=1, bias=False), nn.Tanh())
        self.tail = conv_block_no_relu(base_c, 3)

        # self.alpha = nn.Parameter(torch.tensor(0.01))

    def forward(self, x_cover, x_secret):
        x = torch.cat([x_cover, x_secret], dim=1)

        head = self.head(x)        # 128x128x8
        d1 = self.down1(head)      # 64x64x16
        d2 = self.down2(d1)      # 32x32x32
        d3 = self.down3(d2)      # 16x16x64
        b  = self.bottleneck(d3) # 4x4x128

        up0 = self.upsample(b)

        u1_sc_down4  = self.up1(torch.cat([up0, d3], dim=1))
        up1 = self.upsample(u1_sc_down4)

        up2_sc_down3 = self.up2(torch.cat([up1, d2], dim=1))
        up2 = self.upsample(up2_sc_down3)

        up3_sc_down2 = self.up3(torch.cat([up2, d1], dim=1))
        up3 = self.upsample(up3_sc_down2)

        u4_sc_head = self.up4(torch.cat([up3, head], dim=1))

        return torch.clamp(x_cover + self.tail(u4_sc_head), -1, 1)




In [ ]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(self, base_c=16): # Server có thể chạy base_c=32 hoặc 64 thoải mái
        super().__init__()

        # Khối conv_block sử dụng Tích chập đầy đủ (Full Convolution)
        def conv_block(in_c, out_c, stride=1):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=True),
                nn.BatchNorm2d(out_c),
                nn.LeakyReLU(0.1, inplace=True) # Dùng LeakyReLU tốt hơn cho việc khôi phục
            )

        # --- 1. DOWNSAMPLING PATH (Bóc tách đặc trưng từ ảnh Stego) ---
        self.head = conv_block(3, base_c)               # H x W
        self.down1 = conv_block(base_c, base_c*2, stride=2)    # H/2
        self.down2 = conv_block(base_c*2, base_c*4, stride=2)  # H/4
        self.down3 = conv_block(base_c*4, base_c*8, stride=2)  # H/8
        self.down4 = conv_block(base_c*8, base_c*16, stride=2) # H/16

        # Level 5: Bottleneck sâu nhất
        self.bottleneck = conv_block(base_c*16, base_c*16, stride=2) # H/32

        # --- 2. UPSAMPLING PATH (Tái tạo ảnh Secret) ---
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True) # Bilinear mượt hơn Nearest

        # Up 1: H/32 -> H/16
        self.up1 = conv_block(base_c*16 + base_c*16, base_c*8)
        # Up 2: H/16 -> H/8
        self.up2 = conv_block(base_c*8 + base_c*8, base_c*4)
        # Up 3: H/8 -> H/4
        self.up3 = conv_block(base_c*4 + base_c*4, base_c*2)
        # Up 4: H/4 -> H/2
        self.up4 = conv_block(base_c*2 + base_c*2, base_c)
        # Up 5: H/2 -> H
        self.up5 = conv_block(base_c + base_c, base_c)

        # Output: Trả về ảnh Secret 3 kênh
        self.tail = nn.Sequential(
            nn.Conv2d(base_c, 3, kernel_size=3, padding=1),
            nn.Tanh() # Secret image thường được chuẩn hóa về [-1, 1]
        )

    def forward(self, x_stego):
        # --- Encoder Path (Down) ---
        c1 = self.head(x_stego)
        c2 = self.down1(c1)
        c3 = self.down2(c2)
        c4 = self.down3(c3)
        c5 = self.down4(c4)
        b  = self.bottleneck(c5)

        # --- Decoder Path (Up with Skip Connections) ---
        u1 = self.up1(torch.cat([self.upsample(b), c5], dim=1))
        u2 = self.up2(torch.cat([self.upsample(u1), c4], dim=1))
        u3 = self.up3(torch.cat([self.upsample(u2), c3], dim=1))
        u4 = self.up4(torch.cat([self.upsample(u3), c2], dim=1))
        u5 = self.up5(torch.cat([self.upsample(u4), c1], dim=1))

        return self.tail(u5)

In [ ]:
# ==============================================================================
# 3. DỮ LIỆU & UTILS
# ==============================================================================
from torchmetrics.image.psnr import PeakSignalNoiseRatio
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure

class RealImageDataset(Dataset):
    def __init__(self, real_dir, img_size=128):
        try: self.paths = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.lower().endswith(('png','jpg','jpeg'))]
        except: self.paths = []
        self.t = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor(), transforms.Normalize([0.5]*3,[0.5]*3)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try: return self.t(Image.open(self.paths[i]).convert("RGB"))
        except: return torch.zeros(3, 128, 128)

def get_loaders(cfg):
    dl_train = DataLoader(RealImageDataset(cfg.TRAIN_REAL_DIR, cfg.IMG_SIZE),
                          batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

    dl_val = DataLoader(RealImageDataset(cfg.VAL_REAL_DIR, cfg.IMG_SIZE),
                        batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, drop_last=True)

    return dl_train, dl_val

# Metrics
psnr_fn = PeakSignalNoiseRatio(data_range=1.0).to(Config.DEVICE)
ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(Config.DEVICE)
def calc_metrics(i1, i2):
    i1, i2 = ((i1+1)/2).clamp(0,1), ((i2+1)/2).clamp(0,1)
    return psnr_fn(i1, i2).item(), ssim_fn(i1, i2).item()

def denorm(t): return ((t+1)/2).clamp(0,1).cpu().permute(0,2,3,1).numpy()



In [ ]:
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, a=0.1)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def train_step(models, opt_G, losses, data, cfg):
    # Chỉ còn Encoder và Decoder
    enc, dec = models
    l1, l2 = losses
    cover, secret = data

    # --- Train G (Encoder + Decoder) ---
    opt_G.zero_grad()

    # 1. Giấu tin (Encode)
    stego = enc(cover, secret)

    # 2. Khôi phục tin (Decode)
    rev = dec(stego)

    # --- Tính toán Loss ---
    l_c = cfg.LAMBDA_COVER_L1 * l1(stego, cover) + cfg.LAMBDA_COVER_L2 * l2(stego, cover)

    # Secret Loss (Mục tiêu: Rev giống hệt Secret)
    l_s = cfg.LAMBDA_SECRET_L1 * l1(rev, secret) + cfg.LAMBDA_SECRET_L2 * l2(rev, secret)


    # Tổng Loss (Đã bỏ Adversarial Loss)
    loss_G = l_c + l_s

    # Cập nhật trọng số
    loss_G.backward()

    torch.nn.utils.clip_grad_norm_(
        list(enc.parameters()) + list(dec.parameters()), 1.0
    )

    opt_G.step()

    # Trả về các chỉ số để log (Bỏ các chỉ số của Discriminator)
    return {
        'Loss_G': loss_G.item(), # Tổng loss G
        'Cov': l_c.item(),       # Loss ảnh bìa
        'Sec': l_s.item(),       # Loss tin mật
        'Adv': 0.0,              # Gán cứng bằng 0 để không phá vỡ format log cũ
        'D_Acc': 0.0             # Gán cứng bằng 0
    }

In [ ]:
def run_validation(models, loader, cfg, epoch):
    enc, dec, disc = models

    enc.eval()
    dec.eval()

    acc = {'Cov':0, 'Sec':0, 'PSNR_C':0, 'SSIM_C':0, 'PSNR_S':0, 'SSIM_S':0}
    cnt = 0

    with torch.no_grad():
        for cover in loader:
            cover = cover.to(cfg.DEVICE)

            # Secret random từng batch
            secret = cover[torch.randperm(cover.size(0))].to(cfg.DEVICE)

            # Forward
            stego = enc(cover, secret)
            rev   = dec(stego)

            # Reconstruction loss
            acc['Cov'] += nn.L1Loss()(stego, cover).item()
            acc['Sec'] += nn.L1Loss()(rev, secret).item()

            # PSNR + SSIM
            pc, sc = calc_metrics(stego, cover)
            ps, ss = calc_metrics(rev, secret)

            acc['PSNR_C'] += pc
            acc['SSIM_C'] += sc
            acc['PSNR_S'] += ps
            acc['SSIM_S'] += ss
            cnt += 1

    # Trung bình toàn bộ batch
    res = {k: v/cnt for k, v in acc.items()}

    # --- 2. Visualize Images (Lưu ảnh mẫu) ---
    c = next(iter(loader)).to(cfg.DEVICE)[:3]
    s = c[torch.randperm(c.size(0))].to(cfg.DEVICE)
    with torch.no_grad():
        st, re = enc(c, s), dec(enc(c, s))
        diff = torch.abs(st - c)
        diff = (diff - diff.min()) / (diff.max() - diff.min() + 1e-8)

    imgs = [denorm(c), denorm(s), denorm(st), denorm(diff), denorm(re)]
    titles = ["Cover", "Secret", "Stego", "Residual", "Revealed"]

    fig, axes = plt.subplots(3, 5, figsize=(15, 9))
    for i in range(3):
        for j in range(5):
            img_show = imgs[j][i]
            if j == 3: img_show = img_show.mean(2)
            axes[i,j].imshow(img_show, cmap='inferno' if j==3 else None)
            if i==0: axes[i,j].set_title(titles[j], fontweight='bold')
            axes[i,j].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, f"vis_epoch_{epoch}.png"))
    plt.close()

    return res



def plot_training_history(history, cfg, epoch):
    if len(history['val_psnr_c']) == 0: return

    ep_range = range(1, len(history['train_loss_g']) + 1)

    if len(ep_range) != len(history['val_psnr_c']):
        min_len = min(len(ep_range), len(history['val_psnr_c']))
        ep_range = range(1, min_len + 1)

    fig, ax = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle(f'Training Progress - Epoch {epoch}', fontsize=16)

    # [0,0] Total Losses
    ax[0,0].plot(ep_range, history['train_loss_g'][:len(ep_range)], label='G Loss', color='b')
    ax[0,0].plot(ep_range, history['train_loss_d'][:len(ep_range)], label='D Loss', color='r', alpha=0.7)
    ax[0,0].set_title('Total Losses')
    ax[0,0].legend(); ax[0,0].grid(True, alpha=0.3)

    # [0,1] Generator Components
    ax[0,1].plot(ep_range, history['train_cov'][:len(ep_range)], label='Cover', color='green')
    ax[0,1].plot(ep_range, history['train_sec'][:len(ep_range)], label='Secret', color='orange')
    ax[0,1].plot(ep_range, history['train_adv'][:len(ep_range)], label='Adv', color='purple')
    ax[0,1].set_title('Generator Components')
    ax[0,1].legend(); ax[0,1].grid(True, alpha=0.3)

    # [0,2] Discriminator Accuracy
    ax[0,2].plot(ep_range, history['train_d_acc'][:len(ep_range)], label='D Acc', color='black')
    ax[0,2].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
    ax[0,2].set_title('Discriminator Accuracy')
    ax[0,2].set_ylim(0, 1.05)
    ax[0,2].legend(); ax[0,2].grid(True, alpha=0.3)

    # [1,0] PSNR
    ax[1,0].plot(ep_range, history['val_psnr_c'][:len(ep_range)], label='Cover', marker='.')
    ax[1,0].plot(ep_range, history['val_psnr_s'][:len(ep_range)], label='Secret', marker='.')
    ax[1,0].set_title('PSNR (dB)')
    ax[1,0].legend(); ax[1,0].grid(True, alpha=0.3)

    # [1,1] SSIM
    ax[1,1].plot(ep_range, history['val_ssim_c'][:len(ep_range)], label='Cover', marker='.')
    ax[1,1].plot(ep_range, history['val_ssim_s'][:len(ep_range)], label='Secret', marker='.')
    ax[1,1].set_title('SSIM')
    ax[1,1].set_ylim(0, 1.05)
    ax[1,1].legend(); ax[1,1].grid(True, alpha=0.3)

    # [1,2] Info
    ax[1,2].axis('off')
    best_loss = min(np.array(history['train_cov']) + np.array(history['train_sec']))
    ax[1,2].text(0.1, 0.5, f"Epoch: {epoch}\nBest Combine Loss: {best_loss:.4f}", fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(cfg.SAVE_DIR, "training_history_full.png"))
    plt.close()

In [ ]:
# ==============================================================================
# 5. MAIN PROGRAM
# ==============================================================================
def main():
    cfg = Config()

    dl_train, dl_val = get_loaders(cfg)
    print(f"Train: {len(dl_train)} | Val: {len(dl_val)}")
    if len(dl_train) == 0: return print("Lỗi Data")

    # Init Models
    enc = Encoder(base_c=cfg.BASE_CHANNEL).to(cfg.DEVICE)
    dec = Decoder(base_c=cfg.BASE_CHANNEL).to(cfg.DEVICE)

    # enc.apply(init_weights)
    # dec.apply(init_weights)

    print(f"--- Params ---")
    print(f"Encoder: {sum(p.numel() for p in enc.parameters()):,}")
    print(f"Decoder: {sum(p.numel() for p in dec.parameters()):,}")
    # print(f"Discrim: {sum(p.numel() for p in disc.parameters()):,}")

    # Optimizers: SRNet học chậm hơn Generator
    opt_G = optim.Adam(list(enc.parameters())+list(dec.parameters()), lr=cfg.LEARNING_RATE, betas=(0.5, 0.999))

    losses = [nn.L1Loss(), nn.MSELoss()]

    # Resume
    start_ep, best_score = 1, float('inf')
    ckpt_path = os.path.join(cfg.SAVE_DIR, "checkpoint.pth")

    history = {
        'train_loss_g': [], 'train_loss_d': [],
        'train_cov': [], 'train_sec': [], 'train_adv': [],
        'train_d_acc': [],
        'val_psnr_c': [], 'val_ssim_c': [],
        'val_psnr_s': [], 'val_ssim_s': []
    }


    if os.path.exists(ckpt_path):
        print(" Đang tải checkpoint...")
        c = torch.load(ckpt_path, map_location=cfg.DEVICE)
        enc.load_state_dict(c['enc']);
        dec.load_state_dict(c['dec']);
        # disc.load_state_dict(c['disc'])
        opt_G.load_state_dict(c['opt_G']);
        start_ep, best_score, history = c['epoch']+1, c['best'], c.get('hist', history)

    print(f"\n BẮT ĐẦU TRAINING (Epoch {start_ep} -> {cfg.NUM_EPOCHS})")

    for epoch in range(start_ep, cfg.NUM_EPOCHS + 1):
        enc.train(); dec.train();
        t0 = time.time()

        running_results = {'Loss_D': 0, 'Loss_G': 0, 'Cov': 0, 'Sec': 0, 'Adv': 0, 'D_Acc': 0}

        for idx, cover in enumerate(dl_train):
            cover = cover.to(cfg.DEVICE)
            # Tạo secret ngẫu nhiên ngay trong vòng lặp
            secret = cover[torch.randperm(cover.size(0))].to(cfg.DEVICE)

            m = train_step((enc, dec), opt_G, losses, (cover, secret), cfg)


            for k, v in m.items(): running_results[k] += v

            if idx % 200 == 0:
                print(f"E{epoch} B{idx:04d} | Loss_G={m['Loss_G']:.4f} | Cov={m['Cov']:.4f} | Sec={m['Sec']:.4f}")


        for k in running_results:
            running_results[k] /= len(dl_train)

        history['train_loss_g'].append(running_results['Loss_G'])
        history['train_loss_d'].append(0.0)
        history['train_cov'].append(running_results['Cov'])
        history['train_sec'].append(running_results['Sec'])
        history['train_adv'].append(0.0)
        history['train_d_acc'].append(0.0)


        # Validation
        print(f">>> Validating...")
        res = run_validation((enc, dec, None), dl_val, cfg, epoch)

        history['val_psnr_c'].append(res['PSNR_C'])
        history['val_ssim_c'].append(res['SSIM_C'])
        history['val_psnr_s'].append(res['PSNR_S'])
        history['val_ssim_s'].append(res['SSIM_S'])


        # Score = Cov + Sec
        score = res['Cov'] + res['Sec']
        print(f"    [RESULT] Score: {score:.4f}")
        print(f"    COVER : PSNR={res['PSNR_C']:.2f}dB | SSIM={res['SSIM_C']:.4f}")
        print(f"    SECRET: PSNR={res['PSNR_S']:.2f}dB | SSIM={res['SSIM_S']:.4f}")

         # --- PLOT HISTORY ---
        try:
            plot_training_history(history, cfg, epoch)
        except Exception as e:
            print(f"⚠ Lỗi vẽ biểu đồ: {e}")


        # Save Checkpoint

        save_dict = {
            'epoch': epoch,
            'best': best_score,
            'hist': history,
            'enc': enc.state_dict(),
            'dec': dec.state_dict(),
            'opt_G': opt_G.state_dict()
        }
        torch.save(save_dict, ckpt_path)

        # Save Best
        if score < best_score:
            print(f"NEW BEST! Saving models...")
            best_score = score
            torch.save(enc.state_dict(), os.path.join(cfg.SAVE_DIR, "best_enc.pth"))
            torch.save(dec.state_dict(), os.path.join(cfg.SAVE_DIR, "best_dec.pth"))
        print("\n")


        # Plot nhanh Loss (Chỉ còn Cov và Sec)
        plt.figure(figsize=(10, 5))
        plt.plot(history['train_cov'], label='Cover Loss (L1+L2+LPIPS)')
        plt.plot(history['train_sec'], label='Secret Loss (L1)')
        plt.title('Training Loss History')
        plt.legend()
        plt.savefig(os.path.join(cfg.SAVE_DIR, "loss_history.png"))
        plt.close()

        print(f"--- Done Epoch {epoch} ({time.time() - t0:.1f}s) ---\n")

if __name__ == "__main__":
    main()

Train: 1562 | Val: 312
--- Params ---
Encoder: 56,966
Decoder: 1,773,363
 Đang tải checkpoint...

 BẮT ĐẦU TRAINING (Epoch 7 -> 40)
E7 B0000 | Loss_G=0.0751 | Cov=0.0072 | Sec=0.0679
E7 B0200 | Loss_G=0.0783 | Cov=0.0087 | Sec=0.0696
E7 B0400 | Loss_G=0.1017 | Cov=0.0089 | Sec=0.0928
E7 B0600 | Loss_G=0.0675 | Cov=0.0086 | Sec=0.0589
E7 B0800 | Loss_G=0.0721 | Cov=0.0080 | Sec=0.0641
E7 B1000 | Loss_G=0.0694 | Cov=0.0086 | Sec=0.0608
E7 B1200 | Loss_G=0.0900 | Cov=0.0074 | Sec=0.0826
E7 B1400 | Loss_G=0.0730 | Cov=0.0085 | Sec=0.0646
>>> Validating...
    [RESULT] Score: 0.0570
    COVER : PSNR=40.33dB | SSIM=0.9809
    SECRET: PSNR=29.04dB | SSIM=0.9078
NEW BEST! Saving models...


--- Done Epoch 7 (179.2s) ---

E8 B0000 | Loss_G=0.0665 | Cov=0.0082 | Sec=0.0584
E8 B0200 | Loss_G=0.0728 | Cov=0.0094 | Sec=0.0634
E8 B0400 | Loss_G=0.0653 | Cov=0.0081 | Sec=0.0572
E8 B0600 | Loss_G=0.0681 | Cov=0.0082 | Sec=0.0599
E8 B0800 | Loss_G=0.0715 | Cov=0.0074 | Sec=0.0642
E8 B1000 | Loss_G=0.05